# BUS 32120, Week 9
# OLS metrics and train/test split

Learning objectives: 
* Regression metrics: R-squared, RMSE
* Regression (OLS) vs classification
* Classification metrics: accuracy, confusion matrix


In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from math import sqrt

In [13]:
print("pandas version ", pd.__version__)
#print("sklearn version ", sklearn.__version__)
print("seaborn version ", sns.__version__)

pandas version  2.1.1
seaborn version  0.12.2


NB: seaborn isn't playing nicely with pandas 2.1, so suppress warnings to avoid having to scroll through a lot of red.

More info on what's causing the warning (as of Oct 2023): https://github.com/mwaskom/seaborn/issues/3486

In [14]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# Regression metrics: R-squared and RMSE

### Revisiting the Ames housing dataset

In [2]:
df = pd.read_csv('./data/ames_housing.csv')

### 1. Clean dataset and create modeling input dataset

In [3]:
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [4]:
# remove outliers 

df2 = df[df['Gr Liv Area'] <= 4000]

In [5]:
# choose only some features for modeling

df3 = df2.loc[ :, ['Lot Area', 'Overall Qual', 'Overall Cond', 
                      'Year Built', 'Year Remod/Add', 'Gr Liv Area', 
                      'Full Bath', 'Bedroom AbvGr', 'Fireplaces', 
                      'Garage Cars','SalePrice']]

df3 = df3.fillna( value = {'Garage Cars' : 0})

In [6]:
# create output vector

y = df3['SalePrice']
X = df3.drop('SalePrice',axis=1)

### 2. Train / test split

In machine learning, you'll often want to split your full dataset into [training and testing subsets](https://machinelearningmastery.com/train-test-split-for-evaluating-machine-learning-algorithms/). This is called a **train/test split**. The procdure for doing the split is to divide your full dataset into training data, on which you build the model, then a testing set on which you report the error metric (e.g. R^2 or RMSE). 

The point of doing a train/test split is to replicate how the trained model would work on data it has not seen before and not been trained on (the holdout or test set).

In [7]:
X_train, X_test, y_train, y_test = train_test_split( X, y, 
                                                    test_size = 0.2, random_state=42)

In [8]:
print("X train = ", X_train.shape)
print("y train = ", y_train.shape)
print("\nX test = ", X_test.shape)
print("y test = ", y_test.shape)

X train =  (2340, 10)
y train =  (2340,)

X test =  (585, 10)
y test =  (585,)


### 3. Run linear regression and calculate metrics

#### 3a. Calculate R-Squared

In [9]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Calculate the r-squared metric for the regression
score = lr_model.score(X_test, y_test)

# Print output
print('\nR-square = ', f'{score:.3f}')
print('Feature coefficients (aka slopes): \n')

for feature, coef in zip(X_train.columns, lr_model.coef_):
    print(feature, ':', f'{coef:.2f}') 


R-square =  0.818
Feature coefficients (aka slopes): 

Lot Area : 1.18
Overall Qual : 19750.20
Overall Cond : 4956.09
Year Built : 534.64
Year Remod/Add : 141.23
Gr Liv Area : 73.67
Full Bath : -8692.91
Bedroom AbvGr : -10641.08
Fireplaces : 5325.00
Garage Cars : 10120.80


### 3b. Calculate RMSE

In [10]:
from math import sqrt 

In [11]:
y_predicted = lr_model.predict(X_test)

In [12]:
rmse = sqrt(mean_squared_error(y_test, y_predicted))
print("RMSE = ", f'{rmse:.2f}')

RMSE =  35941.35


--------------
### Check for understanding: What do these two error metrics tell us? Which one is better, R-squared or RMSE, and why?

---------------------------------------------------

Root 
Mean
Squared
Error